# Kontur Population Australia → PMTiles

This notebook downloads the [Kontur Population dataset for Australia](https://data.humdata.org/dataset/kontur-population-australia) (H3 hexagon grid with population estimates) and converts it to **PMTiles** using **Tippecanoe** for use in web maps.

## 1. Install Dependencies

In [1]:
%pip install geopandas requests pmtiles fiona

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

## 2. Download the Kontur Population Dataset

The dataset is a gzipped GeoPackage from the Humanitarian Data Exchange (HDX).

In [2]:
import os, gzip, shutil, requests
from pathlib import Path

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

DOWNLOAD_URL = "https://geodata-eu-central-1-kontur-public.s3.amazonaws.com/kontur_datasets/kontur_population_AU_20231101.gpkg.gz"
GZ_PATH = DATA_DIR / "kontur_population_AU.gpkg.gz"
GPKG_PATH = DATA_DIR / "kontur_population_AU.gpkg"

if not GZ_PATH.exists():
    print(f"Downloading {DOWNLOAD_URL} ...")
    resp = requests.get(DOWNLOAD_URL, stream=True, timeout=300)
    resp.raise_for_status()
    total = int(resp.headers.get("content-length", 0))
    downloaded = 0
    with open(GZ_PATH, "wb") as f:
        for chunk in resp.iter_content(chunk_size=1 << 20):
            f.write(chunk)
            downloaded += len(chunk)
            if total:
                print(f"\r  {downloaded / 1e6:.1f} / {total / 1e6:.1f} MB", end="")
    print("\nDone.")
else:
    print(f"Already downloaded: {GZ_PATH}")

# Decompress
if not GPKG_PATH.exists():
    print("Decompressing ...")
    with gzip.open(GZ_PATH, "rb") as gz, open(GPKG_PATH, "wb") as out:
        shutil.copyfileobj(gz, out)
    print(f"Decompressed to {GPKG_PATH} ({GPKG_PATH.stat().st_size / 1e6:.1f} MB)")
else:
    print(f"Already decompressed: {GPKG_PATH} ({GPKG_PATH.stat().st_size / 1e6:.1f} MB)")

  41.4 / 41.4 MB
Done.
Decompressing ...
Decompressed to data/kontur_population_AU.gpkg (101.4 MB)


## 3. Explore the Downloaded Data

Check file info and list layers in the GeoPackage.

In [3]:
import fiona

print(f"File size: {GPKG_PATH.stat().st_size / 1e6:.1f} MB")
print(f"\nLayers in GeoPackage:")
for layer in fiona.listlayers(GPKG_PATH):
    with fiona.open(GPKG_PATH, layer=layer) as src:
        print(f"  - {layer}: {len(src)} features, CRS={src.crs['init'] if 'init' in src.crs else src.crs_wkt[:60]}...")

File size: 101.4 MB

Layers in GeoPackage:
  - population: 517689 features, CRS=epsg:3857...


## 4. Load and Inspect with GeoPandas

In [4]:
import geopandas as gpd

gdf = gpd.read_file(GPKG_PATH)
print(f"Shape: {gdf.shape}")
print(f"CRS:   {gdf.crs}")
print(f"\nColumns: {list(gdf.columns)}")
print(f"\nPopulation stats:")
print(gdf["population"].describe())
gdf.head()

Shape: (517689, 3)
CRS:   EPSG:3857

Columns: ['h3', 'population', 'geometry']

Population stats:
count    517689.000000
mean         51.071773
std         319.638312
min           1.000000
25%           1.000000
50%           2.000000
75%           5.000000
max       21673.000000
Name: population, dtype: float64


,h3,population,geometry
0,88e0b6b1b5fffff,1.0,"POLYGON ((8171665.121 -6986645.812, 8171843.06..."
1,88e0b6b1b1fffff,3.0,"POLYGON ((8170633.183 -6987879.168, 8170811.28..."
2,88da7335dbfffff,1.0,"POLYGON ((17681976.59 -7317751.61, 17682021.27..."
3,88da733585fffff,1.0,"POLYGON ((17684770.887 -7311118.564, 17684815...."
4,88da7334e7fffff,1.0,"POLYGON ((17681089.131 -7319071.513, 17681133...."


## 5. Convert to Newline-Delimited GeoJSON

Tippecanoe reads GeoJSON Lines (`.geojsonl`) efficiently — one feature per line, no need to hold everything in memory.

In [5]:
GEOJSONL_PATH = DATA_DIR / "kontur_population_AU.geojsonl"

if not GEOJSONL_PATH.exists():
    print("Converting GeoPackage → GeoJSON Lines ...")
    gdf.to_file(GEOJSONL_PATH, driver="GeoJSONSeq")
    print(f"Written: {GEOJSONL_PATH} ({GEOJSONL_PATH.stat().st_size / 1e6:.1f} MB)")
else:
    print(f"Already exists: {GEOJSONL_PATH} ({GEOJSONL_PATH.stat().st_size / 1e6:.1f} MB)")

Converting GeoPackage → GeoJSON Lines ...
Written: data/kontur_population_AU.geojsonl (181.5 MB)


## 6. Install Tippecanoe (felt fork)

The **felt/tippecanoe** fork supports PMTiles output natively (the original Mapbox version only outputs MBTiles). Install it from source:

In [ ]:
import subprocess

# Check if felt/tippecanoe is already installed (it reports "tippecanoe v2.x.x" with pmtiles support)
result = subprocess.run(["tippecanoe", "--version"], capture_output=True, text=True)
version_str = (result.stderr.strip() or result.stdout.strip())
print(f"Current: {version_str or 'not installed'}")

# Install felt/tippecanoe from source if needed
if not version_str or "v2." not in version_str:
    print("\nInstalling felt/tippecanoe (with PMTiles support) ...")
    cmds = [
        "apt-get update && apt-get install -y build-essential libsqlite3-dev zlib1g-dev",
        "rm -rf /tmp/tippecanoe && git clone https://github.com/felt/tippecanoe.git /tmp/tippecanoe",
        "cd /tmp/tippecanoe && make -j$(nproc) && make install",
    ]
    for cmd in cmds:
        print(f"$ {cmd}")
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        if r.returncode != 0:
            print(f"  stderr: {r.stderr[-300:]}")
            break
    
    result = subprocess.run(["tippecanoe", "--version"], capture_output=True, text=True)
    print(f"\nInstalled: {result.stderr.strip() or result.stdout.strip()}")
else:
    print("✓ felt/tippecanoe already installed")

tippecanoe v1.36.0


## 7. Generate PMTiles with Tippecanoe

Key flags:
- `-Z2 -z12`: zoom range 2–12  
- `--drop-densest-as-needed`: drop features to fit tile size limits at lower zooms  
- `--extend-zooms-if-still-dropping`: keep going if tiles are still too large  
- `-l population`: layer name in the vector tiles

In [18]:
PMTILES_PATH = DATA_DIR / "kontur_population_AU.pmtiles"

# Remove old MBTiles file that was wrongly named .pmtiles
if PMTILES_PATH.exists():
    PMTILES_PATH.unlink()
    print("Removed old file")

# Also remove failed conversion attempt
converted = DATA_DIR / "kontur_population_AU_converted.pmtiles"
if converted.exists():
    converted.unlink()

cmd = [
    "tippecanoe",
    "-o", str(PMTILES_PATH),
    "-Z", "2",
    "-z", "12",
    "--drop-densest-as-needed",
    "--extend-zooms-if-still-dropping",
    "-l", "population",
    "--force",
    str(GEOJSONL_PATH),
]

print(f"Running: {' '.join(cmd)}\n")
result = subprocess.run(cmd, capture_output=True, text=True)
if result.returncode == 0:
    print("Tippecanoe completed successfully!")
    print(result.stderr)
else:
    print(f"Error (exit code {result.returncode}):")
    print(result.stderr)

Removed old file
Running: tippecanoe -o data/kontur_population_AU.pmtiles -Z 2 -z 12 --drop-densest-as-needed --extend-zooms-if-still-dropping -l population --force data/kontur_population_AU.geojsonl

Tippecanoe completed successfully!
Read 0.00 million features
Read 0.01 million features
Read 0.02 million features
Read 0.03 million features
Read 0.04 million features
Read 0.05 million features
Read 0.06 million features
Read 0.07 million features
Read 0.08 million features
Read 0.09 million features
Read 0.10 million features
Read 0.11 million features
Read 0.12 million features
Read 0.13 million features
Read 0.14 million features
Read 0.15 million features
Read 0.16 million features
Read 0.17 million features
Read 0.18 million features
Read 0.19 million features
Read 0.20 million features
Read 0.21 million features
Read 0.22 million features
Read 0.23 million features
Read 0.24 million features
Read 0.25 million features
Read 0.26 million features
Read 0.27 million features
Read 0.2

## 8. Verify the PMTiles Output

In [19]:
import json

if PMTILES_PATH.exists():
    size_mb = PMTILES_PATH.stat().st_size / 1e6
    print(f"File: {PMTILES_PATH} ({size_mb:.1f} MB)\n")

    with open(PMTILES_PATH, "rb") as f:
        magic = f.read(7)

    if magic == b"PMTiles":
        print("✓ Valid PMTiles file!")
        from pmtiles.reader import MmapSource, Reader as PMReader
        with open(PMTILES_PATH, "rb") as f:
            reader = PMReader(MmapSource(f))
            header = reader.header()
            meta = reader.metadata()
            print(f"Min zoom: {header['min_zoom']}")
            print(f"Max zoom: {header['max_zoom']}")
            print(f"Bounds:   [{header['min_lon']:.4f}, {header['min_lat']:.4f}, {header['max_lon']:.4f}, {header['max_lat']:.4f}]")
            if meta:
                print(f"\nMetadata:\n{json.dumps(meta, indent=2)[:500]}")
        print(f"\n→ Download {PMTILES_PATH} and place it in public/kontur_population_AU.pmtiles")
    elif magic[:6] == b"SQLite":
        print("✗ Still MBTiles — felt/tippecanoe may not have installed correctly.")
    else:
        print(f"✗ Unknown format. First bytes: {magic}")
else:
    print("✗ PMTiles file not found.")

File: data/kontur_population_AU.pmtiles (86.1 MB)

✗ Still MBTiles — felt/tippecanoe may not have installed correctly.
